# Aula Consulta
ACA 1 Computación en la Nube
Grupo pendiente de confirmar. Este cuaderno requiere ejecución real contra Qdrant Cloud. Sin resultados remotos simulados.

In [1]:
import subprocess,sys
subprocess.check_call([sys.executable,'-m','pip','install','-r','requirements.txt'])

0

## 1. Credenciales y conexión
Ejecute desde la carpeta del proyecto. La clave se lee desde .env local o se solicita de forma oculta y se carga como variable de entorno. No incluir .env en la entrega.

In [2]:
from pathlib import Path
import os,getpass
from dotenv import load_dotenv
load_dotenv(Path.cwd()/'.env')
if not os.getenv('QDRANT_URL'):
    os.environ['QDRANT_URL']=input('Punto final HTTPS: ').strip()
if not os.getenv('QDRANT_API_KEY'):
    os.environ['QDRANT_API_KEY']=getpass.getpass('Clave API oculta: ')
from proyecto import *
preparar_carpetas()
conectar()
print('Conexión remota confirmada')

Conexión remota confirmada


## 2. Dataset estructurado y SQL
395 registros portugueses. Consultas de resumen, apoyo familiar y tiempo de estudio. Los documentos CUN se vinculan por temas, sin relación individual ni equivalencia normativa.

In [3]:
resumen=base_sql()
resumen

{'resumen': [{'estudiantes': 395, 'promedio_G3': 10.42}],
 'por_apoyo_familiar': [{'famsup': 'no',
   'estudiantes': 153,
   'promedio_G3': 10.64},
  {'famsup': 'yes', 'estudiantes': 242, 'promedio_G3': 10.27}],
 'por_tiempo_estudio': [{'studytime': 1,
   'estudiantes': 105,
   'promedio_G3': 10.05},
  {'studytime': 2, 'estudiantes': 198, 'promedio_G3': 10.17},
  {'studytime': 3, 'estudiantes': 65, 'promedio_G3': 11.4},
  {'studytime': 4, 'estudiantes': 27, 'promedio_G3': 11.26}]}

## 3. Ingesta
Tres PDF del portal CUN. Conservar archivo, URL, página y hash. Revisar texto extraído.

In [4]:
paginas=cargar_paginas()
print('Páginas con texto:',len(paginas))
print(paginas[0]['texto'][:700])

Páginas con texto: 88
Página 1 de 25 ACUERDO No. 019 DE 2023 DEL CONSEJO DIRECTIVO REGLAMENTO ESTUDIANTIL DE LA CORPORACIÓN UNIFICADA NACIONAL DE EDUCACIÓN SUPERIOR LIBRO PRIMERO RÉGIMEN GENERAL DE LOS ESTUDIANTES CAPÍTULO PRIMERO OBJETO Y DE LA CALIDAD DE ESTUDIANTE Art 1. OBJETO DEL REGLAMENTO ESTUDIANTIL: Es objeto del presente Reglamento Estudiantil regular las relaciones académicas, financieras, administrativas y disciplinarias devenidas de la relación entre la Institución y sus estudiantes. En virtud de ello, se establecen los derechos, deberes, estímulos, distinciones y procedimientos asociados de la comunidad estudiantil y, por ende, sin perjuicio de disposiciones especiales al respecto, se aplica a todos


## 4. Chunking
Hasta 220 palabras, solapamiento de 40 y límite de 460 tokens incluyendo prefijo. Respetar páginas. E5 multilingüe usa 384 dimensiones y prefijos passage: / query:. La guía permite modelos alternativos, por lo que se documenta E5 en lugar del ejemplo específico Gemini/RETRIEVAL_QUERY.

In [5]:
m=modelo_local()
fragmentos=fragmentar(paginas,m.tokenizer)
print('Fragmentos:',len(fragmentos),'Máximo tokens:',max(x['tokens'] for x in fragmentos))
fragmentos[0]

C:\Users\ANTERNOL\Downloads\ACA1_Nube_Aula_Consulta_Preparacion\ACA_Nube\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3948.60it/s]

Fragmentos: 183 Máximo tokens: 366


{'archivo': 'reglamento_estudiantil.pdf',
 'url': 'https://repo.cunapp.dev/web/2025/Normativa/reglamento_estudiantil.pdf',
 'pagina': 1,
 'sha256': 'eff0c0989eb224ea0e731f0201a41aa2116d2f3dceaf8a9421e4b2e730b944c8',
 'texto': 'Página 1 de 25 ACUERDO No. 019 DE 2023 DEL CONSEJO DIRECTIVO REGLAMENTO ESTUDIANTIL DE LA CORPORACIÓN UNIFICADA NACIONAL DE EDUCACIÓN SUPERIOR LIBRO PRIMERO RÉGIMEN GENERAL DE LOS ESTUDIANTES CAPÍTULO PRIMERO OBJETO Y DE LA CALIDAD DE ESTUDIANTE Art 1. OBJETO DEL REGLAMENTO ESTUDIANTIL: Es objeto del presente Reglamento Estudiantil regular las relaciones académicas, financieras, administrativas y disciplinarias devenidas de la relación entre la Institución y sus estudiantes. En virtud de ello, se establecen los derechos, deberes, estímulos, distinciones y procedimientos asociados de la comunidad estudiantil y, por ende, sin perjuicio de disposiciones especiales al respecto, se aplica a todos a los estudiantes matriculados en modalidad presencial, v irtual y a dis

## 5. Indexación remota
Distancia coseno, normalización, lotes de 32. IDs estables para el mismo corpus. No elimina colecciones. Cambios de corpus requieren colección nueva.

In [6]:
reporte=indexar(m,fragmentos)
reporte

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Batches:   8%|▊         | 1/12 [00:01<00:11,  1.02s/it]

Batches:  17%|█▋        | 2/12 [00:01<00:09,  1.06it/s]

Batches:  25%|██▌       | 3/12 [00:02<00:08,  1.10it/s]

Batches:  33%|███▎      | 4/12 [00:03<00:07,  1.13it/s]

Batches:  42%|████▏     | 5/12 [00:04<00:07,  1.03s/it]

... (5 outputs left out)

Batches:  92%|█████████▏| 11/12 [00:10<00:00,  1.02it/s]

Batches: 100%|██████████| 12/12 [00:11<00:00,  1.31it/s]

Batches: 100%|██████████| 12/12 [00:11<00:00,  1.09it/s]

{'coleccion': 'aula_consulta_e5_v1',
 'fragmentos_corpus': 183,
 'puntos_remotos': 183,
 'dimension': 384,
 'distancia': 'Cosine',
 'modelo': 'intfloat/multilingual-e5-small',
 'tamano_palabras': 220,
 'solapamiento_palabras': 40,
 'max_tokens': 366,
 'relacion_dataset': 'temática, no individual'}

## 6. Top-3 y puntuaciones
Cuatro consultas educativas y una fuera de tema. Coseno recalculado con vectores. Latencia solo de consulta remota, excluye embeddings.

In [7]:
df=validar(m)
df[['pregunta','rango','similitud','coseno_recalculado','archivo','pagina','latencia_remota_s']]

,pregunta,rango,similitud,coseno_recalculado,archivo,pagina,latencia_remota_s
0,¿Qué derechos tienen los estudiantes en la ins...,1,0.869209,0.869209,reglamento_estudiantil.pdf,26,0.653373
1,¿Qué derechos tienen los estudiantes en la ins...,2,0.860553,0.860553,reglamento_estudiantil.pdf,31,0.653373
2,¿Qué derechos tienen los estudiantes en la ins...,3,0.857684,0.857684,reglamento_estudiantil.pdf,27,0.653373
3,¿Qué acompañamiento académico ofrece bienestar...,1,0.882077,0.882077,politica_bienestar.pdf,18,0.633932
4,¿Qué acompañamiento académico ofrece bienestar...,2,0.876149,0.876149,politica_bienestar.pdf,22,0.633932
5,¿Qué acompañamiento académico ofrece bienestar...,3,0.874228,0.874228,politica_bienestar.pdf,17,0.633932
6,¿Cómo participa la familia en el bienestar de ...,1,0.871641,0.871641,politica_bienestar.pdf,21,0.693083
7,¿Cómo participa la familia en el bienestar de ...,2,0.855327,0.855327,reglamento_bienestar.pdf,13,0.693083
8,¿Cómo participa la familia en el bienestar de ...,3,0.854562,0.854562,reglamento_bienestar.pdf,5,0.693083
9,¿Qué apoyos se contemplan frente a dificultade...,1,0.834433,0.834432,politica_bienestar.pdf,19,0.655327


In [8]:
for f in df.to_dict('records'):
    print('\nPregunta:',f['pregunta'],'Rango:',f['rango'],'Score:',round(f['similitud'],4))
    print(f['archivo'],'página',f['pagina'])
    print(f['texto'])


Pregunta: ¿Qué derechos tienen los estudiantes en la institución? Rango: 1 Score: 0.8692
reglamento_estudiantil.pdf página 26
Ser escuchado, orientado, atendido y asistido por el facilitador, académico, administrativo y directivo de la Institución así como recibir respuesta oportuna a sus requerimientos, para ello el estudiante deberá efectuar todas sus solicitudes de manera respetuosa y atendiendo en todos los eventos los conductos y procedimientos institucionales. 3. Ejercer sin restricción alguna la libertad de expresión, investigación, discusión y de análisis sin que en ningún caso esto implique o justifique la transgresión de derechos o garantías constitucionales, legales, reglamentarias o institucionalmente establecidas a favor miembros de la Institución o terceras personas. 4. Participar activa y const ructivamente en el progreso y devenir de la Institución, para tales efectos el estudiante siempre deberá atender las normas estatutarias y reglamentarias previstas para ello. 5. 

## 7. Evaluación humana
Lea cada fragmento y asigne 0/1 según su relevancia. La similitud no es probabilidad ni precisión. Documente que la consulta ajena carece de evidencia adecuada.

In [9]:
revision=df.copy()
revision['relevante']=None
revision.to_csv(BASE/'resultados/revision_relevancia.csv',index=False)
print('Complete relevante (0/1) en resultados/revision_relevancia.csv.')

Complete relevante (0/1) en resultados/revision_relevancia.csv.


In [10]:
import pandas as pd
r=pd.read_csv(BASE/'resultados/revision_relevancia.csv')
r=r[r['pregunta']!=PREGUNTAS[-1]]
if r['relevante'].isna().any():
    print('Pendiente revisión humana. Sin cálculo de precisión todavía.')
else:
    assert r['relevante'].isin([0,1]).all()
    display(r.groupby('pregunta')['relevante'].mean().rename('precision_at_3'))

Pendiente revisión humana. Sin cálculo de precisión todavía.


## 8. Conclusiones y fuentes
Completar con resultados reales: relevancia, errores, latencia y control fuera de tema. Revisar duplicidades entre documentos. Sin afirmar alta precisión ni costo cero sin comprobar consumo.

Fuentes: fuentes.json; https://doi.org/10.24432/C5TG7T; https://huggingface.co/intfloat/multilingual-e5-small; https://qdrant.tech/documentation/.